# 02. Моделирование, абляции, SHAP и бизнес-стоимость

Сравнение Rules Only, Baseline LogReg, GBDT (Tx Only vs Full Features), Isolation Forest и Full AI Agent на строгом разбиении по времени.

In [ ]:
import json, sys
from pathlib import Path
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

meta = json.loads(Path("../models/artifacts/metadata.json").read_text(encoding="utf-8"))
ab_df = pd.DataFrame(meta["ablations"])
ab_df[[
    "variant", "pr_auc", "roc_auc", "recall_at_1pct_fpr",
    "precision_at_100", "fraud_prevented_amount", "false_positive_cost", "net_savings"
]]

In [ ]:
# Визуализация PR-AUC и Net Savings по вариантам абляций
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ab_df.plot(kind="barh", x="variant", y="pr_auc", ax=axes[0], legend=False, title="PR-AUC (Test Set)")
ab_df.plot(kind="barh", x="variant", y="net_savings", ax=axes[1], legend=False, title="Чистая экономия (Net Savings)", color="green")
plt.tight_layout()
plt.show()

In [ ]:
# Локальное SHAP-объяснение для мошеннической транзакции из тестовой выборки
import joblib
from src.agent.explainer import FraudExplainer
from src.features.online_store import ALL_FEATURE_NAMES
from src.features.engineering import time_based_split

feat_df = pd.read_parquet("../data/processed/features.parquet")
_, _, test_df = time_based_split(feat_df)
gbdt = joblib.load("../models/artifacts/gbdt_model.joblib")
bg = joblib.load("../models/artifacts/background_median.joblib")
explainer = FraudExplainer(gbdt, ALL_FEATURE_NAMES, bg)

sample_fraud = test_df[test_df["isFraud"] == 1].iloc[0]
factors = explainer.explain_single(sample_fraud[ALL_FEATURE_NAMES].to_dict(), top_k=6)
pd.DataFrame(factors)

In [ ]:
# Отчёт по дрейфу данных (PSI: Train vs Test)
pd.DataFrame(meta["train_vs_test_drift"]["features"])